# Food-101 ViT-B/16 acceleration: controlled experiment

This notebook is the presentation layer for the reproducible package in `src/`. It tests whether **TF32/high float32 matmul precision + `torch.compile` + BF16 autocast** reduce training time relative to FP32 eager execution while holding the data, model, optimizer, batch size, workers, seed, and evaluation loop fixed.

The exploratory source notebook recorded 27.71 minutes versus 13.43 minutes (2.06x), but that pair had confounders. The runner below produces the controlled measurement.

## Data and model

Food-101 contains 101 classes, 75,750 training images, and 25,250 test images. Images are resized to 384 x 384, so ViT-B/16 creates `(384/16)^2 = 576` patch tokens plus one class token. The SWAG E2E pretrained model has 12 encoder blocks. Blocks 10–11, final layer normalization, and a new 101-class head are trainable: 14,254,949 of 86,168,165 parameters.

In [ ]:
import platform
import torch
import torchvision

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name())
    print('CUDA runtime:', torch.version.cuda)
    print('cuDNN:', torch.backends.cudnn.version())
    print('BF16 supported:', torch.cuda.is_bf16_supported())

## The only treatment differences

| Switch | Baseline | Accelerated |
|---|---|---|
| float32 matmul policy | `highest` | `high` |
| TF32 | off | on |
| execution | eager | `torch.compile` |
| autocast | off / FP32 | on / BF16 |

Compilation cost is included in the first epoch. The report separately aggregates all epochs (cold) and epochs after the first (steady state).

In [ ]:
from food101_accel.config import ExperimentConfig

baseline = ExperimentConfig(profile='baseline').as_serializable_dict()
accelerated = ExperimentConfig(profile='accelerated').as_serializable_dict()
print('Baseline:', baseline['features'])
print('Accelerated:', accelerated['features'])

## Run the isolated A/B benchmark

`DATA_ROOT` can contain ImageFolder-style `train/` and `test/` directories or receive the official torchvision download. The capped mode validates the pipeline; full measurements omit both batch caps. Two repeats alternate execution order.

In [ ]:
from pathlib import Path
import subprocess
import sys

DATA_ROOT = Path('data')
DOWNLOAD_DATA = True
RUN_BENCHMARK = False  # Execution is disabled by default.
SMOKE_TEST = True

command = [
    sys.executable, '-m', 'food101_accel.benchmark',
    '--data-root', str(DATA_ROOT),
    '--epochs', '3',
    '--repeats', '2',
    '--discard-first-epochs', '1',
]
if DOWNLOAD_DATA:
    command += ['--download']
if SMOKE_TEST:
    command += ['--max-train-batches', '20', '--max-eval-batches', '10']

print('Command:', ' '.join(command))
if RUN_BENCHMARK:
    subprocess.run(command, check=True)

## Inspect the latest result

Capped smoke-test measurements are labeled separately from full-data results. A complete record contains `benchmark.json`, GPU/software metadata, cold and steady ratios, and the predeclared accuracy tolerance.

In [ ]:
import json

reports = sorted(Path('artifacts').glob('*-benchmark/benchmark.json'))
if not reports:
    print('No benchmark report yet.')
else:
    latest = reports[-1]
    report = json.loads(latest.read_text(encoding='utf-8'))
    print('Report:', latest)
    print(json.dumps(report.get('speedups', {}), indent=2))
    for profile, values in report['summary'].items():
        print(profile, {
            'steady_seconds': values['steady_epoch_seconds']['mean'],
            'images_per_second': values['train_samples_per_second']['mean'],
            'final_top1_percent': values['final_eval_top1']['mean'] * 100,
            'final_top5_percent': values['final_eval_top5']['mean'] * 100,
        })